# INPUT MODULE: MARGINALS AND PAIR COPULA

This example showcases how to define a probabilistic input model with a pair copula dependence structure. In order to display the copula density, the marginals are kept as uniform distributions in $[0,1]$. In this way, the joint probabilty density function (PDF) and the copula density are equivalent.

## Package imports

In [ ]:
from uqpylab import sessions

## Start a remote UQCloud session

In [ ]:
# Start the session
mySession = sessions.cloud()
# (Optional) Get a convenient handle to the command line interface
uq = mySession.cli
# Reset the session
mySession.reset()

## Set the random seed for reproducibility

In [ ]:
uq.rng(100, 'twister');

## Probabilistic input model

The probabilistic input model consists of two variables:

$X_1 \sim \mathcal{U}(0, 1)$

and 

$X_2 \sim \mathcal{U}(0, 1)$

In [ ]:
InputOpts = {
    'Marginals': [
        {
            'Type': 'Uniform',
            'Parameters': [0,1]
        },
        {
            'Type': 'Uniform',
            'Parameters': [0,1]
        }
    ]
}

Define the copula between the two variables as a Gumbel pair copula:

$C_{12} = \text{Gumbel}(1.5)$

In [ ]:
InputOpts['Copula'] = {
    'Type': 'Pair',
    'Family': 'Gumbel',
    'Parameters': 1.5
}

Create an INPUT object based on the specified marginals and copula:

In [ ]:
myInput = uq.createInput(InputOpts)

Print a report of the INPUT object:

In [ ]:
uq.print(myInput)

Display a visualization of the INPUT object:

In [ ]:
uq.display(myInput);

Alternatively, the copula of the input model can be specified using the function `uq_PairCopula(Type, Parameters, Rotation)` as follows:

In [ ]:
InputOpts['Copula'] = uq.PairCopula('Gumbel', 1.5)

Create the INPUT object:

In [ ]:
myInput2 = uq.createInput(InputOpts)

Display a visualization of the INPUT object:

In [ ]:
uq.display(myInput2);

## Dependence properties of the input model

The dependence properties of the input model are fully determined by the input copula.

For instance, a popular measure of the global dependence between two random variables is their Kendall's tau, defined as the probability that two realizations from the random variables are concordant minus the probability that they are discordant.


In [ ]:
Tau_K = uq.PairCopulaKendallTau(myInput['Copula'])

The probability of joint extremes is also of interest, for instance in reliability and fragility analysis. The Gumbel copula defined above models the upper tail dependence, that is, a positive probability $\lambda_u$ that the random variables it couples take jointly high values:

In [ ]:
Lambda_U = uq.PairCopulaUpperTailDep(myInput['Copula'])

This makes the Gumbel copula different from the Gaussian copula, which instead never assigns upper or lower tail dependence, even for high values of its correlation parameter:

In [ ]:
GaussianCopula = uq.PairCopula('Gaussian',0.99)
Lambda_U_Gaussian = uq.PairCopulaUpperTailDep(GaussianCopula)

Different parametric pair copula families have different dependence properties (Kendall's tau, upper/lower tail dependence). These properties should be considered when deciding which pair copula to use to model the input! For summary, refer to the UQLab's Input Manual, Chapter "Theory".

## Copula rotation

The PDF of a copula distribution can be rotated by $90$, $180$, or $270$ degrees to model different types of dependencies. For instance:

In [ ]:
InputOpts['Copula'] = {
    'Type': 'Pair',
    'Family': 'Gumbel',
    'Parameters': 1.5,
    'Rotation': 180
}

creates a version of the Gumbel copula rotated by $180$ degrees. Mathematically, this is obtained by flipping the original copula density $c(u,v)$ around both axes: $c_{180}(u,v) = c(1-u,1-v)$.

Create an INPUT object based on the rotated copula:

In [ ]:
myInput_rot180 = uq.createInput(InputOpts)

Display a visualization of the object:

In [ ]:
uq.display(myInput_rot180);

This new copula has different dependence properties. For instance, it has no upper tail dependence anymore:

In [ ]:
uq.PairCopulaUpperTailDep(myInput_rot180['Copula'])

but has lower tail one:

In [ ]:
uq.PairCopulaLowerTailDep(myInput_rot180['Copula'])

Analogously, the copula PDF can be rotated by $90$ and $270$ degrees:

In [ ]:
InputOpts['Copula']['Rotation'] = 90

myInput_rot90 = uq.createInput(InputOpts)

uq.display(myInput_rot90);

In [ ]:
InputOpts['Copula']['Rotation'] = 270

myInput_rot270 = uq.createInput(InputOpts)

uq.display(myInput_rot270);

## Terminate the remote UQCloud session

In [ ]:
mySession.quit()